# activity recognition with a hidden markov model

guessing standing, walking, jumping or still from phone accelerometer and gyroscope data.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)

RATE = 50          # hz
WINDOW = 64        # samples per window
STEP = 32          # half overlap
N_SYMBOLS = 12

ACTIVITIES = ["standing", "walking", "jumping", "still"]
state_index = {a: i for i, a in enumerate(ACTIVITIES)}

## 1. loading

one csv per recording, named `activity_person_number.csv`. the label comes from the file name.

In [ ]:
def load_folder(folder):
    out = []
    for path in sorted(glob.glob(os.path.join(folder, "*.csv"))):
        name = os.path.basename(path).lower()
        label = next(a for a in ACTIVITIES if name.startswith(a))
        df = pd.read_csv(path)
        df.columns = [c.strip().lower() for c in df.columns]
        out.append((label, df.dropna().reset_index(drop=True)))
    return out


train = load_folder("data/train")
for a in ACTIVITIES:
    print(a, sum(1 for lab, _ in train if lab == a), "files")

## 2. features

each window becomes 14 numbers: movement stats, fft features, and the tilt of the phone.

tilt matters because standing and still are both motionless and only differ in orientation.

In [ ]:
def corr(a, b):
    return 0.0 if min(a.std(), b.std()) < 1e-8 else float(np.corrcoef(a, b)[0, 1])


def window_features(w):
    ax, ay, az = w["ax"].values, w["ay"].values, w["az"].values
    acc = np.sqrt(ax**2 + ay**2 + az**2)
    gyro = np.sqrt(w["gx"].values**2 + w["gy"].values**2 + w["gz"].values**2)

    spectrum = np.abs(np.fft.rfft(acc - acc.mean()))
    freqs = np.fft.rfftfreq(len(acc), d=1.0 / RATE)
    dominant = freqs[np.argmax(spectrum[1:]) + 1]       # bin 0 is just the average
    power = spectrum**2 / (np.sum(spectrum**2) + 1e-12)
    entropy = -np.sum(power[power > 0] * np.log(power[power > 0]))

    tilt = np.array([ax.mean(), ay.mean(), az.mean()]) / (acc.mean() + 1e-8)

    return [acc.mean(), acc.std(), np.mean(np.abs(ax) + np.abs(ay) + np.abs(az)),
            corr(ax, ay), corr(ax, az), corr(ay, az),
            gyro.mean(), gyro.std(),
            dominant, np.log1p(spectrum.var()), entropy, *tilt]


def build(recordings):
    out = []
    for label, df in recordings:
        rows = [window_features(df.iloc[i:i + WINDOW])
                for i in range(0, len(df) - WINDOW + 1, STEP)]
        if rows:
            out.append({"label": label, "features": np.array(rows)})
    return out


train_seqs = build(train)
all_features = np.vstack([s["features"] for s in train_seqs])
print("feature matrix", all_features.shape)

the features are on very different scales, so z-score them using the training data only.

In [ ]:
mean = all_features.mean(axis=0)
std = np.where(all_features.std(axis=0) < 1e-8, 1.0, all_features.std(axis=0))
normalize = lambda x: (x - mean) / std

## 3. observation symbols

the model needs discrete observations, so k-means turns each feature vector into a cluster number.

In [ ]:
def kmeans(data, k, iterations=100):
    centers = data[np.random.choice(len(data), k, replace=False)].copy()
    for _ in range(iterations):
        assign = np.argmin(np.linalg.norm(data[:, None] - centers[None], axis=2), axis=1)
        new = np.array([data[assign == c].mean(axis=0) if np.any(assign == c) else centers[c]
                        for c in range(k)])
        if np.allclose(new, centers):
            break
        centers = new
    return centers


def to_symbols(features):
    x = normalize(features)
    return np.argmin(np.linalg.norm(x[:, None] - centers[None], axis=2), axis=1)


centers = kmeans(normalize(all_features), N_SYMBOLS)
for s in train_seqs:
    s["symbols"] = to_symbols(s["features"])

print(train_seqs[0]["label"], train_seqs[0]["symbols"][:20])

## 4. the model

hidden states are the four activities, observations are the symbols from the features.

### which algorithm and why

viterbi gives one activity per window, which is exactly what our per window metrics need.

baum-welch learns the matrices but unsupervised it shuffles the states, so we count from labels first.

forward and backward are scaled and viterbi runs in log space, otherwise long sequences underflow.

In [ ]:
class HMM:
    def __init__(self, n_states, n_symbols):
        self.pi = np.full(n_states, 1.0 / n_states)
        self.A = np.full((n_states, n_states), 1.0 / n_states)
        self.B = np.full((n_states, n_symbols), 1.0 / n_symbols)

    def forward(self, obs):
        alpha = np.zeros((len(obs), len(self.pi)))
        scale = np.zeros(len(obs))
        alpha[0] = self.pi * self.B[:, obs[0]]
        scale[0] = alpha[0].sum() + 1e-300
        alpha[0] /= scale[0]
        for t in range(1, len(obs)):
            alpha[t] = (alpha[t - 1] @ self.A) * self.B[:, obs[t]]
            scale[t] = alpha[t].sum() + 1e-300
            alpha[t] /= scale[t]
        return alpha, scale

    def backward(self, obs, scale):
        beta = np.zeros((len(obs), len(self.pi)))
        beta[-1] = 1.0 / scale[-1]
        for t in range(len(obs) - 2, -1, -1):
            beta[t] = (self.A @ (self.B[:, obs[t + 1]] * beta[t + 1])) / scale[t]
        return beta

    def viterbi(self, obs):
        logA, logB = np.log(self.A + 1e-300), np.log(self.B + 1e-300)
        delta = np.log(self.pi + 1e-300) + logB[:, obs[0]]
        back = []
        for t in range(1, len(obs)):
            scores = delta[:, None] + logA
            back.append(np.argmax(scores, axis=0))
            delta = scores.max(axis=0) + logB[:, obs[t]]

        path = [int(np.argmax(delta))]
        for b in reversed(back):
            path.append(int(b[path[-1]]))
        return np.array(path[::-1])

    def baum_welch(self, sequences, iterations=15, tol=1e-4):
        previous = None
        for it in range(iterations):
            pi_acc = np.zeros_like(self.pi)
            A_acc = np.zeros_like(self.A)
            B_acc = np.zeros_like(self.B)
            total = 0.0

            for obs in sequences:
                alpha, scale = self.forward(obs)
                beta = self.backward(obs, scale)
                total += np.sum(np.log(scale))

                gamma = alpha * beta
                gamma /= gamma.sum(axis=1, keepdims=True) + 1e-300
                pi_acc += gamma[0]

                for t in range(len(obs) - 1):
                    xi = (alpha[t][:, None] * self.A
                          * self.B[:, obs[t + 1]][None, :] * beta[t + 1][None, :])
                    A_acc += xi / (xi.sum() + 1e-300)
                    B_acc[:, obs[t]] += gamma[t]
                B_acc[:, obs[-1]] += gamma[-1]

            self.pi = pi_acc / pi_acc.sum()
            self.A = A_acc / A_acc.sum(axis=1, keepdims=True)
            self.B = B_acc / B_acc.sum(axis=1, keepdims=True)

            print("iteration", it + 1, "log likelihood", round(total, 2))
            if previous is not None and abs(total - previous) < tol:
                print("converged")
                break
            previous = total

each recording is one activity, so we stitch them into sessions to learn the transitions.

In [ ]:
n = len(ACTIVITIES)
pi_counts, A_counts = np.ones(n), np.ones((n, n))     # start at 1 so nothing is zero
B_counts = np.ones((n, N_SYMBOLS))

for s in train_seqs:
    i = state_index[s["label"]]
    pi_counts[i] += 1
    A_counts[i, i] += len(s["symbols"]) - 1
    for sym in s["symbols"]:
        B_counts[i, sym] += 1

sessions = []
order = list(range(len(train_seqs)))
for _ in range(10):
    np.random.shuffle(order)
    picked = order[:4]
    labels = [train_seqs[j]["label"] for j in picked]
    sessions.append(np.concatenate([train_seqs[j]["symbols"] for j in picked]))
    for a, b in zip(labels, labels[1:]):
        A_counts[state_index[a], state_index[b]] += 1

model = HMM(n, N_SYMBOLS)
model.pi = pi_counts / pi_counts.sum()
model.A = A_counts / A_counts.sum(axis=1, keepdims=True)
model.B = B_counts / B_counts.sum(axis=1, keepdims=True)
model.baum_welch(sessions)

## 5. transition matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(model.A, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(n), ACTIVITIES, rotation=45)
ax.set_yticks(range(n), ACTIVITIES)
ax.set_xlabel("to")
ax.set_ylabel("from")
for i in range(n):
    for j in range(n):
        ax.text(j, i, round(model.A[i, j], 2), ha="center", va="center",
                color="white" if model.A[i, j] > 0.5 else "black", fontsize=9)
plt.tight_layout()
plt.savefig("figures/transition_matrix.png", dpi=120)
plt.show()

## 6. unseen data

the test files come from a separate session that was never used in training.

In [ ]:
test_seqs = build(load_folder("data/test"))
for s in test_seqs:
    s["symbols"] = to_symbols(s["features"])

truth = np.concatenate([np.full(len(s["symbols"]), state_index[s["label"]]) for s in test_seqs])
predicted = model.viterbi(np.concatenate([s["symbols"] for s in test_seqs]))

plt.figure(figsize=(11, 3))
plt.step(range(len(truth)), truth, where="post", label="actual", linewidth=2)
plt.step(range(len(predicted)), predicted, where="post", label="predicted", linestyle="--")
plt.yticks(range(n), ACTIVITIES)
plt.xlabel("window")
plt.legend()
plt.tight_layout()
plt.savefig("figures/decoded_test.png", dpi=120)
plt.show()

In [ ]:
cm = np.zeros((n, n), dtype=int)
for t, p in zip(truth, predicted):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(n), ACTIVITIES, rotation=45)
ax.set_yticks(range(n), ACTIVITIES)
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
for i in range(n):
    for j in range(n):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.savefig("figures/confusion_matrix.png", dpi=120)
plt.show()

In [ ]:
rows = []
total = cm.sum()
for i, activity in enumerate(ACTIVITIES):
    tp = cm[i, i]
    fn, fp = cm[i].sum() - tp, cm[:, i].sum() - tp
    tn = total - tp - fn - fp
    rows.append({"state (activity)": activity,
                 "number of samples": int(cm[i].sum()),
                 "sensitivity": round(tp / (tp + fn), 3),
                 "specificity": round(tn / (tn + fp), 3),
                 "overall accuracy": round((tp + tn) / total, 3)})

results = pd.DataFrame(rows)
print("accuracy across all windows", round(np.trace(cm) / total, 3))
results.to_csv("report/results_table.csv", index=False)
results